In [13]:
import numpy as np
import PIL as Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms


In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5 ), (0.5, 0.5, 0.5))
    ])


In [3]:
train_data = torchvision.datasets.CIFAR10(root='./data', train=True,  transform=transform, download=True)
test_data = torchvision.datasets.CIFAR10(root='./data', train=False, transform=transform, download=True)

train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=32, shuffle=False, num_workers=2)


100%|██████████| 170M/170M [18:57<00:00, 150kB/s]  


In [4]:
image, label = train_data[0]

In [5]:
image.size()

torch.Size([3, 32, 32])

In [ ]:
class_names = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [6]:
class NeuralNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 12, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(12, 24, 5)
        self.fc1 = nn.Linear(24 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) 
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [7]:
net =  NeuralNet()
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [8]:
for epoch in range(30):
    print(f"Traing epoch {epoch }...")

    running_loss = 0.0
    for i, data in enumerate(train_loader, 0):
        inputs, labels = data

        optimizer.zero_grad()

        outputs = net(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'loss: {running_loss / len(train_loader)} .4f')
    


Traing epoch 0...
loss: 2.2548762713955215 .4f
Traing epoch 1...
loss: 1.8210371037896291 .4f
Traing epoch 2...
loss: 1.5552414737293832 .4f
Traing epoch 3...
loss: 1.414242272985645 .4f
Traing epoch 4...
loss: 1.305897477187183 .4f
Traing epoch 5...
loss: 1.2164372193912474 .4f
Traing epoch 6...
loss: 1.1443907222683736 .4f
Traing epoch 7...
loss: 1.0825255644756178 .4f
Traing epoch 8...
loss: 1.031210442574758 .4f
Traing epoch 9...
loss: 0.9843780806029522 .4f
Traing epoch 10...
loss: 0.9429944306326004 .4f
Traing epoch 11...
loss: 0.9027799770417155 .4f
Traing epoch 12...


KeyboardInterrupt: 

In [10]:
torch.save(net.state_dict(), './trained_net.pth')

In [11]:
net = NeuralNet()
net.load_state_dict(torch.load('./trained_net.pth'))

<All keys matched successfully>

In [12]:
correct = 0
total = 0

net.eval()


with torch.no_grad():
    for data in test_loader:
        images, labels = data
        outputs = net(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy {accuracy} %')

Accuracy 65.49 %


In [14]:
new_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])


def load_image(image_path):
    image = Image.open(image_path)
    image = new_transform(image)
    image = image.unsqueeze(0)  
    return image

image_path = ['example1.jpg','example2.jpg']  # Replace with the actual path to your image
images = [load_image(img) for img in image_path]


net.eval()

with torch.no_grad():
    for  image in images:
        output = net(image)
        _, predicted = torch.max(output, 1)
        print(f'Prediction: {class_names[predicted.item()]}')

AttributeError: module 'PIL' has no attribute 'open'